# Maritime Small Object Detection: RF-DETR → DINOv3 Custom Build

Three-path notebook for UAV maritime small object detection on **SeaDronesSee** and **AFO** datasets.

---

## Architecture Trade-off Guide

| | **RF-DETR** (§0A) | **DINOv3 Custom** (§0B) | **DINOv2 Custom** (§1–15) |
|---|---|---|---|
| **Backbone** | DINOv2 ViT-S/14 (fixed) | DINOv3 ViT-S→G (your choice) | DINOv2 ViT-S/14 |
| **Backbone quality** | ★★★☆ | ★★★★★ | ★★★★☆ |
| **Architecture** | LW-DETR (interleaved window+global attn) | DETR-style head (this notebook) | DETR-style head (this notebook) |
| **Setup effort** | `pip install rfdetr` + 3 lines | Medium — build + wire head | Same as DINOv3 |
| **Time to first mAP** | ~1 hr on 4090 | ~4–8 hrs | ~4–8 hrs |
| **Architectural control** | ✗ Black box | ✓ Full | ✓ Full |
| **Custom loss / head** | ✗ Hard to modify | ✓ Trivial | ✓ Trivial |
| **Multi-scale features** | ✓ Built-in (LW-DETR neck) | ✓ `get_intermediate_layers` | ✓ Manual block tapping |
| **Small object perf.** | Strong (>60 mAP COCO) | Best available backbone | Strong |
| **Params (base variant)** | 29 M | ~22 M + head | ~22 M + head |
| **ONNX / TRT export** | ✓ Built-in `model.export()` | Manual `torch.onnx.export` | Manual `torch.onnx.export` |
| **Air-gapped friendly** | ✓ weights downloadable | ✓ `.safetensors` via HF Hub | ✓ `torch.hub` cache |
| **Gram anchoring** | ✗ | ✓ (prevents feature degradation on long runs) | ✗ |
| **Best for** | Baseline + production | Research / ablations / SOTA | Reference / comparison |

### When to use which

- **Start with RF-DETR (§0A)** — get a mAP number on your dataset in an hour. Use it as the benchmark everything else must beat.
- **Move to DINOv3 (§0B)** — when you want to modify the head, change the loss, or experiment with the backbone depth. DINOv3's Gram anchoring makes it meaningfully better than DINOv2 on non-ImageNet domains like open-ocean UAV imagery.
- **Use DINOv2 (§1–15)** — as a controlled ablation against DINOv3 to quantify exactly how much Gram anchoring helps on your specific dataset.

### Key architectural difference: LW-DETR vs this notebook's head

RF-DETR uses **LW-DETR's interleaved window/global attention** inside the ViT encoder — cheap local attention fires every block, expensive global attention fires every 4th block. This is where most of RF-DETR's efficiency comes from at inference time. The custom head in §0B/§1–15 uses standard full self-attention in the encoder, which is simpler but ~2× slower at the same resolution. If you need edge deployment on RHEL hardware, RF-DETR's ONNX export path is the lower-friction route.

---

| Path | When to use | Time to first number |
|------|-------------|----------------------|
| **0A — RF-DETR fine-tune** | Get a strong baseline fast; production deployment | ~1 hr on RTX 4090 |
| **0B — DINOv3 custom build** | Research; full architectural control; ablations | ~4–8 hrs |
| **1–15 — DINOv2 custom build** | Reference implementation; compare against DINOv3 | same as 0B |

> **Recommendation**: Run **0A** first to get a mAP baseline on your dataset, then use **0B / Sections 1–15** to iterate on architecture.

### Key Challenges
- Small objects on complex sea-surface backgrounds (waves, sun glare, reflections)
- UAV viewpoint with extreme scale variation
- Lightweight inference for edge deployment on RHEL hardware


# Maritime Small Object Detection: RF-DETR → DINOv3 Custom Build

Three-path notebook for UAV maritime small object detection on **SeaDronesSee** and **AFO** datasets.

| Path | When to use | Time to first number |
|------|-------------|----------------------|
| **0A — RF-DETR fine-tune** | Get a strong baseline fast; production deployment | ~1 hr on RTX 4090 |
| **0B — DINOv3 custom build** | Research; full architectural control; ablations | ~4–8 hrs |
| **1–15 — DINOv2 custom build** | Reference implementation; compare against DINOv3 | same as 0B |

> **Recommendation**: Run **0A** first to get a mAP baseline on your dataset, then use **0B / Sections 1–15** to iterate on architecture.

### Key Challenges
- Small objects on complex sea-surface backgrounds (waves, sun glare, reflections)
- UAV viewpoint with extreme scale variation
- Lightweight inference for edge deployment on RHEL hardware

In [1]:
# ── Install all dependencies (Python 3.12 compatible) ─────────────────────────
# Run this cell first. Safe to re-run — pip skips already-installed packages.
import sys
!{sys.executable} -m pip install -q \
    torch torchvision \
    "timm>=1.0.20" \
    scipy numpy pillow tqdm \
    matplotlib opencv-python-headless \
    packaging pycocotools \
    pandas huggingface_hub safetensors

import torch
print(f'torch {torch.__version__}')
print(f'CUDA:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

torch 2.10.0+cpu
CUDA:  False


---
# SECTION 0A — Fast Baseline: RF-DETR Fine-tuning

RF-DETR (Roboflow, ICLR 2026) is a production-grade DINOv2 + LW-DETR architecture that is `pip`-installable and fine-tune-optimised. It is the fastest path to a strong mAP number on SeaDronesSee / AFO with minimal boilerplate.

**Variants available:**
- `RFDETRNano` — lightest, best for edge
- `RFDETRBase` — 29 M params, 60.5 mAP on COCO @ 728 px (recommended baseline)
- `RFDETRLarge` — 128 M params, highest accuracy

**Dataset format required:** COCO JSON. SeaDronesSee is already in this format. AFO (YOLO) needs conversion — see the helper cell below.

**Expected directory structure:**
```
dataset/
  train/
    _annotations.coco.json
    img001.jpg ...
  valid/
    _annotations.coco.json
    img001.jpg ...
  test/
    _annotations.coco.json
    img001.jpg ...
```

In [2]:
# ── 0A-1  Install RF-DETR (run once) ─────────────────────────────────────────
# Requires Python >= 3.10
# !pip install rfdetr supervision
# Optional: W&B / TensorBoard logging
# !pip install "rfdetr[metrics]"

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM  : {vram:.1f} GB")

Device: cpu


In [3]:
# ── 0A-2  AFO YOLO → COCO conversion helper ───────────────────────────────────
# Skip this cell if your dataset is already in COCO format (e.g. SeaDronesSee).

import json
import os
from pathlib import Path
from PIL import Image

AFO_CLASSES = ["human", "surfer", "kayak", "boat", "buoy", "sailboat"]

def yolo_to_coco(img_dir: Path, label_dir: Path, out_json: Path,
                 class_names: list) -> None:
    """
    Convert a YOLO-format split (images + .txt labels) to a COCO JSON file
    compatible with RF-DETR's expected _annotations.coco.json layout.
    """
    categories = [{"id": i, "name": n, "supercategory": "object"}
                  for i, n in enumerate(class_names)]
    images, annotations = [], []
    ann_id = 0

    img_paths = sorted(img_dir.glob("*.jpg")) + sorted(img_dir.glob("*.png"))
    for img_id, img_path in enumerate(img_paths):
        w, h = Image.open(img_path).size
        images.append({
            "id": img_id,
            "file_name": img_path.name,
            "width": w, "height": h
        })
        lbl = label_dir / (img_path.stem + ".txt")
        if not lbl.exists():
            continue
        with open(lbl) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:])
                # YOLO normalised → COCO absolute (x1,y1,w,h)
                abs_w  = bw * w
                abs_h  = bh * h
                abs_x1 = (cx - bw / 2) * w
                abs_y1 = (cy - bh / 2) * h
                annotations.append({
                    "id": ann_id, "image_id": img_id,
                    "category_id": cls,
                    "bbox": [abs_x1, abs_y1, abs_w, abs_h],
                    "area": abs_w * abs_h,
                    "iscrowd": 0
                })
                ann_id += 1

    coco = {"images": images, "annotations": annotations,
            "categories": categories}
    out_json.parent.mkdir(parents=True, exist_ok=True)
    with open(out_json, "w") as f:
        json.dump(coco, f)
    print(f"Wrote {len(images)} images, {len(annotations)} anns → {out_json}")


# Example — run for each split:
# AFO_ROOT = Path("./data/afo")
# RFDETR_AFO = Path("./data/afo_coco")
# for split in ["train", "val", "test"]:
#     out_split = "valid" if split == "val" else split  # RF-DETR uses 'valid'
#     yolo_to_coco(
#         img_dir   = AFO_ROOT / "images" / split,
#         label_dir = AFO_ROOT / "labels" / split,
#         out_json  = RFDETR_AFO / out_split / "_annotations.coco.json",
#         class_names = AFO_CLASSES
#     )
#     # Copy images to RFDETR_AFO / out_split /
#     import shutil
#     for p in (AFO_ROOT / "images" / split).glob("*"):
#         shutil.copy(p, RFDETR_AFO / out_split / p.name)

print("YOLO→COCO converter defined.")

YOLO→COCO converter defined.


In [4]:
# ── 0A-3  RF-DETR fine-tuning ─────────────────────────────────────────────────
#
# Batch size guidance (total effective batch = batch_size × grad_accum_steps = 16):
#   RTX 4090 (24 GB)  → batch_size=16, grad_accum_steps=1
#   RTX 3080 (10 GB)  → batch_size=4,  grad_accum_steps=4
#   T4      (16 GB)   → batch_size=4,  grad_accum_steps=4
#
# Resolution must be divisible by 56.  560 is a good small-object sweet spot;
# 728 maximises mAP at the cost of ~40% more compute.

# from rfdetr import RFDETRBase   # or RFDETRNano / RFDETRLarge

# DATASET_DIR  = Path("./data/seadronessee")   # or ./data/afo_coco after conversion
# OUTPUT_DIR   = Path("./checkpoints/rfdetr_sds")

# model = RFDETRBase(resolution=560)           # 560 recommended for small objects

# training_history = []
# def log_epoch(data):
#     training_history.append(data)
#     print(data)                              # prints mAP50, loss, lr each epoch
# model.callbacks["on_fit_epoch_end"].append(log_epoch)

# model.train(
#     dataset_dir     = str(DATASET_DIR),
#     epochs          = 50,
#     batch_size      = 16,          # adjust per GPU (see note above)
#     grad_accum_steps= 1,
#     lr              = 1e-4,
#     output_dir      = str(OUTPUT_DIR),
#     tensorboard     = True,        # remove if rfdetr[metrics] not installed
# )

print("RF-DETR training cell ready.")
print("Uncomment and set DATASET_DIR to run.")

RF-DETR training cell ready.
Uncomment and set DATASET_DIR to run.


In [5]:
# ── 0A-4  RF-DETR inference & evaluation ──────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image as PILImage

# ── Load checkpoint and run inference on a single image ───────────────────────
# from rfdetr import RFDETRBase
# import supervision as sv

# model = RFDETRBase(pretrain_weights=str(OUTPUT_DIR / "checkpoint.pth"))
# img   = PILImage.open("data/seadronessee/test/001.jpg")
# detections = model.predict(img, threshold=0.4)

# # Visualise with supervision
# annotated = sv.BoxAnnotator().annotate(img, detections)
# annotated = sv.LabelAnnotator().annotate(annotated, detections)
# plt.figure(figsize=(10, 6))
# plt.imshow(annotated)
# plt.axis("off")
# plt.title("RF-DETR inference")
# plt.show()

# ── Plot training history ──────────────────────────────────────────────────────
# if training_history:
#     epochs = [d["epoch"] for d in training_history]
#     map50  = [d.get("mAP_50", 0) for d in training_history]
#     loss   = [d.get("train_loss", 0) for d in training_history]
#     fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
#     ax1.plot(epochs, map50);  ax1.set(title="mAP@50", xlabel="epoch")
#     ax2.plot(epochs, loss);   ax2.set(title="Train Loss", xlabel="epoch")
#     plt.tight_layout(); plt.show()

# ── ONNX export (for RHEL edge deployment) ────────────────────────────────────
# model.export(output_dir=str(OUTPUT_DIR))   # writes model.onnx

print("RF-DETR inference / export cell ready.")

RF-DETR inference / export cell ready.


---
# SECTION 0B — Research Path: DINOv3 Custom Backbone

DINOv3 (Meta AI, arXiv 2508.10104, August 2025) scales SSL pre-training to 7B-parameter ViT models on 1.7B images. The key architectural addition over DINOv2 is **Gram anchoring** — a regularisation that prevents dense feature maps from degrading during long training schedules. This makes DINOv3 the strongest available frozen backbone for small object detection, especially on non-ImageNet domains like aerial/maritime imagery.

**Weights availability:** DINOv3 is on HuggingFace and supported by `timm >= 1.0.20`. We load it via `timm` rather than `torch.hub` (DINOv2 path) since that is the documented interface.

**Drop-in compatibility:** The `DINOv3Backbone` below is a direct replacement for `DINOv2Backbone` from Section 3 — same output API `(p3, p4, p5)`, same downstream head. You can swap backbones in `DINOv2Detector` by changing a single argument.

> **Air-gapped note:** Download weights once with `timm.models.hub.download_cached_file` on a networked machine, then copy the `.safetensors` file to your air-gapped environment and point `timm` at the local path via `TIMM_LOCAL_DIR`.

In [6]:
# ── 0B-1  Install / verify timm version ───────────────────────────────────────
# DINOv3 requires timm >= 1.0.20
# !pip install "timm>=1.0.20"

try:
    import timm
    print(f"timm version: {timm.__version__}")
    from packaging import version
    if version.parse(timm.__version__) < version.parse("1.0.20"):
        print("WARNING: timm < 1.0.20 — DINOv3 may not be available. Run: pip install 'timm>=1.0.20'")
    else:
        print("timm version OK for DINOv3.")
except ImportError:
    print("timm not installed. Run: pip install 'timm>=1.0.20'")

timm version: 1.0.25
timm version OK for DINOv3.


In [7]:
# ── 0B-2  DINOv3 backbone (drop-in for DINOv2Backbone) ───────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F

class DINOv3Backbone(nn.Module):
    """
    DINOv3 backbone via timm, with multi-scale feature extraction.

    Produces the same (p3, p4, p5) output tuple as DINOv2Backbone so it is a
    drop-in replacement for the DINOv2Detector head in Sections 4–15.

    Available model names (timm >= 1.0.20):
        'vit_small_patch14_dinov3'   — ~22 M params (recommended for ablations)
        'vit_base_patch14_dinov3'    — ~86 M params
        'vit_large_patch14_dinov3'   — ~307 M params
        'vit_giant2_patch14_dinov3'  — ~1.1 B params

    Gram anchoring (the key DINOv3 innovation) is baked into the pretrained
    weights — no changes needed here at inference/fine-tune time.

    Output feature channels:
        ViT-S: 384   ViT-B: 768   ViT-L: 1024   ViT-G: 1536
    """

    EMBED_DIMS = {
        "vit_small_patch14_dinov3":  384,
        "vit_base_patch14_dinov3":   768,
        "vit_large_patch14_dinov3": 1024,
        "vit_giant2_patch14_dinov3":1536,
    }

    def __init__(self, model_name: str = "vit_small_patch14_dinov3",
                 out_channels: int  = 256,
                 freeze_blocks: int = 8,
                 img_size: int      = 560):   # 560 aligns with RF-DETR default
        super().__init__()
        import timm

        self.patch_size  = 14
        self.in_channels = self.EMBED_DIMS[model_name]

        # Load pretrained DINOv3 from timm
        # features_only=True returns intermediate block outputs as a list
        self.vit = timm.create_model(
            model_name,
            pretrained=True,
            img_size=img_size,
            features_only=False,   # we use get_intermediate_layers manually
        )
        self.num_blocks = len(self.vit.blocks)

        # Tap at 1/4, 1/2, and full depth
        self.tap_indices = [
            self.num_blocks // 4 - 1,
            self.num_blocks // 2 - 1,
            self.num_blocks - 1,
        ]

        self._freeze_blocks(freeze_blocks)

        n = self.in_channels
        self.proj3 = nn.Sequential(
            nn.Conv2d(n, out_channels, 1), nn.BatchNorm2d(out_channels), nn.GELU())
        self.proj4 = nn.Sequential(
            nn.Conv2d(n, out_channels, 1), nn.BatchNorm2d(out_channels), nn.GELU())
        self.proj5 = nn.Sequential(
            nn.Conv2d(n, out_channels, 3, stride=2, padding=1),
            nn.BatchNorm2d(out_channels), nn.GELU())

        self.num_patches_h = img_size // self.patch_size
        self.num_patches_w = img_size // self.patch_size

    def _freeze_blocks(self, n: int):
        for param in self.vit.patch_embed.parameters():
            param.requires_grad = False
        for i, block in enumerate(self.vit.blocks):
            if i < n:
                for param in block.parameters():
                    param.requires_grad = False

    def forward(self, x: torch.Tensor):
        B, _, H, W = x.shape
        H_p = self.num_patches_h
        W_p = self.num_patches_w

        # get_intermediate_layers API note:
        #   timm >= 1.0.14 : n= accepts a list of block indices
        #   reshape=True   : returns [B, C, H, W] directly (no manual reshape needed)
        feats = self.vit.get_intermediate_layers(
            x,
            n=self.tap_indices,       # list of ints — block indices
            reshape=True,             # [B, C, H_p, W_p] directly
            return_class_token=False,
        )

        # feats is already [B, C, H_p, W_p] when reshape=True
        f3, f4, f5 = feats[0], feats[1], feats[2]

        p3 = self.proj3(f3)
        p4 = self.proj4(f4)
        p5 = self.proj5(f5)

        return p3, p4, p5


print("DINOv3Backbone defined.")

DINOv3Backbone defined.


In [8]:
# ── 0B-3  DINOv3 detector (swap backbone into existing head) ──────────────────
# DINOv2DetectionHead is defined in Section 4 — run that cell first, then
# come back here (or run all cells top-to-bottom).

class DINOv3Detector(nn.Module):
    """
    DINOv3 backbone + the same DETR-style head used by DINOv2Detector.
    Swap comment in build_*_model() calls below to compare apples-to-apples.
    """
    def __init__(self, num_classes: int,
                 backbone_name: str = "vit_small_patch14_dinov3",
                 out_channels: int  = 256,
                 num_queries: int   = 100,
                 freeze_blocks: int = 8,
                 img_size: int      = 560):
        super().__init__()
        self.backbone = DINOv3Backbone(
            model_name    = backbone_name,
            out_channels  = out_channels,
            freeze_blocks = freeze_blocks,
            img_size      = img_size,
        )
        # DINOv2DetectionHead is defined in Section 4 — already in scope.
        self.head = DINOv2DetectionHead(
            num_classes = num_classes,
            d_model     = out_channels,
            num_queries = num_queries,
        )

    def forward(self, x):
        p3, p4, p5 = self.backbone(x)
        return self.head(p3, p4, p5)


def build_sds_dinov3_model(device=DEVICE):
    SDS_CLASSES = ["swimmer", "boat", "jetski", "life_saving_appliances", "buoy"]
    return DINOv3Detector(num_classes=len(SDS_CLASSES)).to(device)

def build_afo_dinov3_model(device=DEVICE):
    AFO_CLASSES = ["human", "surfer", "kayak", "boat", "buoy", "sailboat"]
    return DINOv3Detector(num_classes=len(AFO_CLASSES)).to(device)


print("DINOv3Detector defined.")
print("To train: use the train() function from Section 7 with build_sds_dinov3_model().")

DINOv3Detector defined.
To train: use the train() function from Section 7 with build_sds_dinov3_model().


In [9]:
# ── 0B-4  DINOv3 smoke test ───────────────────────────────────────────────────
# Verifies backbone output shapes without needing dataset.
# Requires internet on first run to download ~100 MB of weights.

# import torch
# backbone = DINOv3Backbone("vit_small_patch14_dinov3", out_channels=256, img_size=560)
# backbone.eval()
# x = torch.randn(1, 3, 560, 560)
# with torch.no_grad():
#     p3, p4, p5 = backbone(x)
# print(f"P3: {tuple(p3.shape)}"   # expect [1, 256, 40, 40]
#       f"  P4: {tuple(p4.shape)}"  # expect [1, 256, 40, 40]
#       f"  P5: {tuple(p5.shape)}") # expect [1, 256, 20, 20]

# total = sum(p.numel() for p in backbone.parameters())
# trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
# print(f"Backbone total: {total/1e6:.1f} M  trainable: {trainable/1e6:.1f} M")

print("DINOv3 smoke test cell ready — uncomment to run.")

DINOv3 smoke test cell ready — uncomment to run.


In [10]:
# ── 0B-5  Air-gapped weight download helper ───────────────────────────────────
# Run this on a networked machine, transfer the .safetensors file,
# then set TIMM_LOCAL_DIR on the air-gapped host before importing timm.

def download_dinov3_weights(model_name: str = "vit_small_patch14_dinov3",
                            save_dir: str   = "./weights") -> str:
    """
    Download DINOv3 weights from HuggingFace Hub to a local directory.
    Returns the local path for use with timm's pretrained_cfg override.
    """
    from huggingface_hub import hf_hub_download
    import timm, os

    cfg = timm.get_pretrained_cfg(model_name)
    repo_id = cfg.hf_hub_id        # e.g. 'facebookresearch/dinov3'
    filename = cfg.hf_hub_filename # e.g. 'vit_small_patch14.safetensors'

    local_path = hf_hub_download(
        repo_id  = repo_id,
        filename = filename,
        local_dir= save_dir,
    )
    print(f"Downloaded to: {local_path}")
    return local_path


def load_dinov3_airgapped(model_name: str, weights_path: str,
                          img_size: int = 560):
    """
    Load DINOv3 from a local weights file (air-gapped environment).
    """
    import timm
    model = timm.create_model(
        model_name,
        pretrained     = False,
        img_size       = img_size,
    )
    import safetensors.torch
    state = safetensors.torch.load_file(weights_path)
    model.load_state_dict(state, strict=False)
    print(f"Loaded {model_name} from {weights_path}")
    return model


# Networked machine:
# path = download_dinov3_weights("vit_small_patch14_dinov3", "./weights")

# Air-gapped RHEL:
# vit = load_dinov3_airgapped("vit_small_patch14_dinov3",
#                              "/mnt/transfer/vit_small_patch14.safetensors")

print("Air-gapped weight helpers defined.")

Air-gapped weight helpers defined.


---
# SECTIONS 1–15 — DINOv2 Custom Build (Reference / Ablation)

The remainder of the notebook is the full from-scratch DINOv2 pipeline. Use it to:
- Understand architectural choices before modifying DINOv3Detector
- Run controlled ablations (backbone frozen depth, query count, loss weights)
- Establish a DINOv2 vs DINOv3 comparison on the same head architecture

All class and function definitions below are also used by Section 0B — run cells top-to-bottom for everything to be in scope.

## 1. Environment Setup

In [11]:
# Install dependencies (run once)
# !pip install torch torchvision transformers pycocotools scipy timm
# !pip install matplotlib pillow opencv-python-headless tqdm
# !pip install huggingface_hub

import os
import json
import math
import random
import shutil
import urllib.request
from pathlib import Path
from collections import defaultdict

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.ops as ops
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cpu


## 2. Dataset Download & Structure

### SeaDronesSee
- **Homepage**: https://seadronessee.cs.uni-tuebingen.de/
- 14,227 RGB UAV images; 5 classes: Swimmer, Boat, Jetski, Life-saving Appliances, Buoy
- Annotations in COCO format

### AFO (Aerial Dataset of Floating Objects)
- **Homepage**: https://github.com/JarekCh/AFO_dataset
- 3,647 UAV images; 6 classes: human, surfer, kayak, boat, buoy, sailboat
- Annotations in YOLO format

In [12]:
# ─── Dataset paths ───────────────────────────────────────────────────────────
# Paths match the output of restructure_afo.py
# SDS path will be updated once 7z extraction completes.
#
# Current structure on disk:
#   ~/data/afo_clean/
#     images/  train/  val/  test/
#     labels/  train/  val/  test/
#   ~/data/seadronessee/          ← update after 7z x sds-dataset.zip

from pathlib import Path

DATA_ROOT = Path.home() / "data"
SDS_ROOT  = DATA_ROOT / "seadronessee"   # update once extracted
AFO_ROOT  = DATA_ROOT / "afo_clean"      # restructure_afo.py output

# SeaDronesSee class definitions
SDS_CLASSES  = ["swimmer", "boat", "jetski", "life_saving_appliances", "buoy"]
SDS_CLASS2ID = {c: i for i, c in enumerate(SDS_CLASSES)}

# AFO class definitions (6-category scheme)
AFO_CLASSES  = ["human", "surfer", "kayak", "boat", "buoy", "sailboat"]
AFO_CLASS2ID = {c: i for i, c in enumerate(AFO_CLASSES)}

print("SeaDronesSee classes:", SDS_CLASSES)
print("AFO classes:         ", AFO_CLASSES)
print(f"AFO root exists: {AFO_ROOT.exists()}")
if AFO_ROOT.exists():
    for split in ['train', 'val', 'test']:
        imgs = list((AFO_ROOT / 'images' / split).glob('*.jpg'))
        lbls = list((AFO_ROOT / 'labels' / split).glob('*.txt'))
        print(f"  {split:5}: {len(imgs)} images, {len(lbls)} labels")


SeaDronesSee classes: ['swimmer', 'boat', 'jetski', 'life_saving_appliances', 'buoy']
AFO classes:          ['human', 'surfer', 'kayak', 'boat', 'buoy', 'sailboat']


### 2.1 SeaDronesSee Dataset — COCO Format

In [13]:
class SeaDronesSeeDataset(Dataset):
    """
    SeaDronesSee dataset loader (COCO-format annotations).
    Returns:
        image  : FloatTensor [3, H, W] normalised to ImageNet stats
        target : dict with keys
                   'boxes'  FloatTensor [N, 4]  (cx, cy, w, h) normalised [0,1]
                   'labels' LongTensor  [N]
                   'image_id' int
    """
    def __init__(self, root: Path, split: str = "train",
                 img_size: int = 640, augment: bool = False):
        self.root     = root
        self.split    = split
        self.img_size = img_size
        self.augment  = augment

        ann_file = root / "annotations" / f"{split}.json"
        with open(ann_file) as f:
            coco = json.load(f)

        # Build image-id → annotations mapping
        self.images = {img["id"]: img for img in coco["images"]}
        self.img_ids = [img["id"] for img in coco["images"]]

        # Map COCO category_id → 0-based index
        cat_ids = sorted([c["id"] for c in coco["categories"]])
        self.catid2idx = {cid: i for i, cid in enumerate(cat_ids)}

        self.ann_by_imgid = defaultdict(list)
        for ann in coco["annotations"]:
            self.ann_by_imgid[ann["image_id"]].append(ann)

        # ImageNet normalisation
        self.normalize = T.Normalize(
            mean=[0.485, 0.456, 0.406],
            std =[0.229, 0.224, 0.225]
        )

    def __len__(self):
        return len(self.img_ids)

    def _load_image(self, img_meta):
        path = self.root / "images" / self.split / img_meta["file_name"]
        img  = Image.open(path).convert("RGB")
        return img

    def _augment(self, img, boxes):
        """Simple horizontal flip augmentation."""
        if random.random() > 0.5:
            img = T.functional.hflip(img)
            if len(boxes):
                boxes[:, 0] = 1.0 - boxes[:, 0]  # flip cx
        return img, boxes

    def __getitem__(self, idx):
        img_id   = self.img_ids[idx]
        img_meta = self.images[img_id]
        img      = self._load_image(img_meta)
        orig_w, orig_h = img.size

        # Resize to square
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)

        # Build target (cx, cy, w, h) normalised
        anns = self.ann_by_imgid.get(img_id, [])
        boxes, labels = [], []
        for ann in anns:
            x, y, w, h = ann["bbox"]  # COCO: top-left x,y,w,h (pixels)
            cx = (x + w / 2) / orig_w
            cy = (y + h / 2) / orig_h
            nw = w / orig_w
            nh = h / orig_h
            if nw > 0 and nh > 0:
                boxes.append([cx, cy, nw, nh])
                labels.append(self.catid2idx[ann["category_id"]])

        boxes  = torch.tensor(boxes,  dtype=torch.float32) if boxes  else torch.zeros((0, 4))
        labels = torch.tensor(labels, dtype=torch.long)    if labels else torch.zeros(0, dtype=torch.long)

        img_t = T.functional.to_tensor(img)
        if self.augment:
            img_t, boxes = self._augment(img_t, boxes)
        img_t = self.normalize(img_t)

        return img_t, {"boxes": boxes, "labels": labels, "image_id": img_id}


def sds_collate(batch):
    images, targets = zip(*batch)
    return torch.stack(images), list(targets)


print("SeaDronesSeeDataset class defined.")

SeaDronesSeeDataset class defined.


### 2.2 AFO Dataset — YOLO Format

In [14]:
class AFODataset(Dataset):
    """
    AFO dataset loader (YOLO .txt annotations).
    Each label file has lines: class_id  cx  cy  w  h  (all normalised 0-1)
    """
    def __init__(self, root: Path, split: str = "train",
                 img_size: int = 640, augment: bool = False):
        self.img_size = img_size
        self.augment  = augment

        img_dir   = root / "images" / split
        label_dir = root / "labels" / split

        self.samples = []
        for img_path in sorted(img_dir.glob("*.jpg")) + \
                        sorted(img_dir.glob("*.png")):
            lbl_path = label_dir / (img_path.stem + ".txt")
            if lbl_path.exists():
                self.samples.append((img_path, lbl_path))

        self.normalize = T.Normalize(
            mean=[0.485, 0.456, 0.406],
            std =[0.229, 0.224, 0.225]
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, lbl_path = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)

        boxes, labels = [], []
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls, cx, cy, w, h = map(float, parts)
                    if w > 0 and h > 0:
                        boxes.append([cx, cy, w, h])
                        labels.append(int(cls))

        boxes  = torch.tensor(boxes,  dtype=torch.float32) if boxes  else torch.zeros((0, 4))
        labels = torch.tensor(labels, dtype=torch.long)    if labels else torch.zeros(0, dtype=torch.long)

        img_t = T.functional.to_tensor(img)
        if self.augment and random.random() > 0.5:
            img_t = T.functional.hflip(img_t)
            if len(boxes):
                boxes[:, 0] = 1.0 - boxes[:, 0]
        img_t = self.normalize(img_t)

        return img_t, {"boxes": boxes, "labels": labels, "image_id": idx}


print("AFODataset class defined.")

AFODataset class defined.


### 2.3 Demo: Load & Visualise a Sample

In [15]:
def unnormalise(tensor):
    """Reverse ImageNet normalisation for display."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)


def visualise_sample(img_tensor, target, class_names, title=""):
    """Draw bounding boxes (cx,cy,w,h normalised) on an unnormalised image."""
    img_np = unnormalise(img_tensor).permute(1, 2, 0).numpy()
    H, W   = img_np.shape[:2]

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.imshow(img_np)

    COLORS = plt.cm.get_cmap("tab10", len(class_names))
    for box, label in zip(target["boxes"], target["labels"]):
        cx, cy, w, h = box.numpy()
        x1 = (cx - w / 2) * W
        y1 = (cy - h / 2) * H
        bw, bh = w * W, h * H
        color = COLORS(label.item())
        rect  = patches.Rectangle((x1, y1), bw, bh,
                                   linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, class_names[label.item()],
                color=color, fontsize=8, weight="bold")

    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


# ── Uncomment once dataset paths are configured ──────────────────────────────
# sds_train = SeaDronesSeeDataset(SDS_ROOT, split="train", augment=True)
# img, tgt  = sds_train[0]
# visualise_sample(img, tgt, SDS_CLASSES, title="SeaDronesSee sample")

# afo_train = AFODataset(AFO_ROOT, split="train", augment=True)
# img, tgt  = afo_train[0]
# visualise_sample(img, tgt, AFO_CLASSES, title="AFO sample")

print("Visualisation helper defined.")

Visualisation helper defined.


## 3. DINOv2 Backbone

DINOv2 (Meta AI) is a ViT pre-trained with self-supervised DINO on 142M images. It produces high-quality patch-level features without task-specific supervision — ideal as a frozen or lightly fine-tuned backbone for small object detection.

We extract **multi-scale** features by tapping intermediate ViT blocks (similar to the multi-scale strategy in RT-DETR's encoder).

In [16]:
class DINOv2Backbone(nn.Module):
    """
    DINOv2 ViT-S/14 backbone with multi-scale feature extraction.

    Taps intermediate transformer blocks at 1/4, 1/2, and 3/4 depth
    to produce a coarse feature pyramid analogous to P3/P4/P5.

    Output shapes (input 640×640, patch_size=14):
        P3  : [B, 384, 46, 46]   (stride 14 ≈ 1/14 of input)
        P4  : [B, 384, 46, 46]   — projected to different channels
        P5  : [B, 384, 46, 46]

    Note: Unlike CNNs, ViT produces a single spatial resolution. We simulate
    multi-scale by projecting different depth features through strided convs.
    """

    SCALE_FACTORS = {
        "vits14": 384,
        "vitb14": 768,
        "vitl14": 1024,
        "vitg14": 1536,
    }

    def __init__(self, model_name: str = "vits14",
                 out_channels: int = 256,
                 freeze_blocks: int = 8,    # freeze first N transformer blocks
                 img_size: int = 640):
        super().__init__()
        self.patch_size   = 14
        self.in_channels  = self.SCALE_FACTORS[model_name]
        self.out_channels = out_channels

        # Load from torch.hub (downloads ~80 MB for vits14)
        self.vit = torch.hub.load(
            "facebookresearch/dinov2",
            f"dinov2_{model_name}",
            pretrained=True
        )
        self.num_blocks = len(self.vit.blocks)

        # Tap at quarter-depth intervals
        self.tap_indices = [
            self.num_blocks // 4,
            self.num_blocks // 2,
            self.num_blocks - 1,
        ]

        # Freeze early blocks
        self._freeze_blocks(freeze_blocks)

        n = self.in_channels
        # Project each tapped scale to out_channels
        # P3: shallow → high resolution, use 1×1
        # P4: mid-depth, keep spatial, use 1×1
        # P5: deep semantic, downsample 2×
        self.proj3 = nn.Sequential(
            nn.Conv2d(n, out_channels, 1), nn.BatchNorm2d(out_channels), nn.GELU())
        self.proj4 = nn.Sequential(
            nn.Conv2d(n, out_channels, 1), nn.BatchNorm2d(out_channels), nn.GELU())
        self.proj5 = nn.Sequential(
            nn.Conv2d(n, out_channels, 3, stride=2, padding=1),
            nn.BatchNorm2d(out_channels), nn.GELU())

        # Compute number of patches for reshape
        self.num_patches_h = img_size // self.patch_size
        self.num_patches_w = img_size // self.patch_size

    def _freeze_blocks(self, n):
        """Freeze first n transformer blocks + patch embed."""
        for param in self.vit.patch_embed.parameters():
            param.requires_grad = False
        for i, block in enumerate(self.vit.blocks):
            if i < n:
                for param in block.parameters():
                    param.requires_grad = False

    def _tokens_to_spatial(self, tokens, B):
        """Reshape [B, num_patches, C] → [B, C, H, W]."""
        H, W = self.num_patches_h, self.num_patches_w
        return tokens.permute(0, 2, 1).reshape(B, -1, H, W)

    def forward(self, x):
        B, _, H_img, W_img = x.shape
        # Recompute patch grid from actual input size (handles any resolution)
        self.num_patches_h = H_img // self.patch_size
        self.num_patches_w = W_img // self.patch_size

        # Patch embed
        tokens = self.vit.prepare_tokens_with_masks(x, None)
        # tokens shape: [B, 1 + num_patches, C]  (CLS + patch tokens)

        tapped = {}
        for i, block in enumerate(self.vit.blocks):
            tokens = block(tokens)
            if i in self.tap_indices:
                tapped[i] = tokens[:, 1:, :]  # drop CLS token

        f3 = self._tokens_to_spatial(tapped[self.tap_indices[0]], B)
        f4 = self._tokens_to_spatial(tapped[self.tap_indices[1]], B)
        f5 = self._tokens_to_spatial(tapped[self.tap_indices[2]], B)

        p3 = self.proj3(f3)   # [B, C, H, W]
        p4 = self.proj4(f4)
        p5 = self.proj5(f5)   # [B, C, H/2, W/2]

        return p3, p4, p5


print("DINOv2Backbone class defined.")
print("Note: The backbone will download ~80 MB of weights on first use.")

DINOv2Backbone class defined.
Note: The backbone will download ~80 MB of weights on first use.


## 4. Detection Head

A lightweight DETR-style detection head with:
- **Transformer encoder** to refine multi-scale features
- **N learnable object queries**
- **FFN** predicting class logits + box coordinates

In [17]:
class PositionalEncoding2D(nn.Module):
    """Learned 2D positional encoding added to flattened spatial features."""
    def __init__(self, d_model: int, max_h: int = 64, max_w: int = 64):
        super().__init__()
        self.row_embed = nn.Embedding(max_h, d_model // 2)
        self.col_embed = nn.Embedding(max_w, d_model // 2)

    def forward(self, feat):
        """feat: [B, C, H, W] → pos: [H*W, B, C]"""
        B, C, H, W = feat.shape
        rows = torch.arange(H, device=feat.device)
        cols = torch.arange(W, device=feat.device)
        row_emb = self.row_embed(rows).unsqueeze(1).expand(H, W, -1)  # H,W,C/2
        col_emb = self.col_embed(cols).unsqueeze(0).expand(H, W, -1)
        pos = torch.cat([row_emb, col_emb], dim=-1)                   # H,W,C
        pos = pos.permute(2, 0, 1).unsqueeze(0).expand(B, -1, -1, -1) # B,C,H,W
        return pos


class MLP(nn.Module):
    """Simple feed-forward network used in DETR prediction head."""
    def __init__(self, in_dim, hidden_dim, out_dim, num_layers):
        super().__init__()
        layers = []
        for i in range(num_layers):
            d_in  = in_dim    if i == 0 else hidden_dim
            d_out = out_dim   if i == num_layers - 1 else hidden_dim
            layers.append(nn.Linear(d_in, d_out))
            if i < num_layers - 1:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class DINOv2DetectionHead(nn.Module):
    """
    Lightweight DETR-style detection head on top of DINOv2 multi-scale features.

    Pipeline:
      1. Fuse P3/P4/P5 via upsampling + addition
      2. Flatten + add positional encoding
      3. Transformer encoder (self-attention)
      4. N queries cross-attend to encoded memory
      5. FFN → (class logits, box cxcywh)
    """
    def __init__(self, num_classes: int, d_model: int = 256,
                 num_queries: int = 100, nhead: int = 8,
                 num_enc_layers: int = 2, num_dec_layers: int = 3,
                 dim_feedforward: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.num_queries = num_queries
        self.d_model     = d_model
        self.num_classes = num_classes

        # Feature fusion convolutions (align P4, P5 to P3 resolution)
        self.fuse_p4 = nn.Conv2d(d_model, d_model, 1)
        self.fuse_p5 = nn.Conv2d(d_model, d_model, 1)

        self.pos_enc = PositionalEncoding2D(d_model)

        enc_layer   = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward, dropout, batch_first=False)
        self.encoder = nn.TransformerEncoder(enc_layer, num_enc_layers)

        dec_layer   = nn.TransformerDecoderLayer(
            d_model, nhead, dim_feedforward, dropout, batch_first=False)
        self.decoder = nn.TransformerDecoder(dec_layer, num_dec_layers)

        self.query_embed = nn.Embedding(num_queries, d_model)

        # Prediction heads
        self.class_head = nn.Linear(d_model, num_classes + 1)   # +1 for "no object"
        self.bbox_head  = MLP(d_model, d_model, 4, num_layers=3)

        self._init_weights()

    def _init_weights(self):
        nn.init.uniform_(self.bbox_head.net[-1].bias.data[:2], -0.01, 0.01)

    def _fuse_scales(self, p3, p4, p5):
        """Upsample P4 and P5 to P3 resolution and add."""
        p4_up = F.interpolate(self.fuse_p4(p4), size=p3.shape[-2:], mode="bilinear", align_corners=False)
        p5_up = F.interpolate(self.fuse_p5(p5), size=p3.shape[-2:], mode="bilinear", align_corners=False)
        return p3 + p4_up + p5_up

    def forward(self, p3, p4, p5):
        B = p3.shape[0]

        # 1. Multi-scale fusion
        fused = self._fuse_scales(p3, p4, p5)  # [B, C, H, W]

        # 2. Positional encoding + flatten to sequence
        pos   = self.pos_enc(fused)             # [B, C, H, W]
        src   = (fused + pos).flatten(2).permute(2, 0, 1)  # [HW, B, C]

        # 3. Encode
        memory = self.encoder(src)              # [HW, B, C]

        # 4. Decode with learnable queries
        queries = self.query_embed.weight.unsqueeze(1).expand(-1, B, -1)  # [Q, B, C]
        tgt     = torch.zeros_like(queries)
        hs      = self.decoder(tgt, memory)     # [Q, B, C]
        hs      = hs.permute(1, 0, 2)           # [B, Q, C]

        # 5. Predict
        class_logits = self.class_head(hs)              # [B, Q, num_classes+1]
        bbox_pred    = self.bbox_head(hs).sigmoid()     # [B, Q, 4] — (cx,cy,w,h) in [0,1]

        return class_logits, bbox_pred


print("DINOv2DetectionHead class defined.")

DINOv2DetectionHead class defined.


## 5. Full Model

In [18]:
class DINOv2Detector(nn.Module):
    """End-to-end maritime small object detector: DINOv2 backbone + DETR-style head."""

    def __init__(self, num_classes: int,
                 backbone_name: str = "vits14",
                 out_channels: int  = 256,
                 num_queries: int   = 100,
                 freeze_blocks: int = 8,
                 img_size: int      = 640):
        super().__init__()
        self.backbone = DINOv2Backbone(
            model_name    = backbone_name,
            out_channels  = out_channels,
            freeze_blocks = freeze_blocks,
            img_size      = img_size,
        )
        self.head = DINOv2DetectionHead(
            num_classes = num_classes,
            d_model     = out_channels,
            num_queries = num_queries,
        )

    def forward(self, x):
        p3, p4, p5        = self.backbone(x)
        class_logits, bboxes = self.head(p3, p4, p5)
        return class_logits, bboxes  # [B,Q,C+1], [B,Q,4]


def build_sds_model(device=DEVICE, img_size=560):
    model = DINOv2Detector(num_classes=len(SDS_CLASSES), img_size=img_size).to(device)
    return model

def build_afo_model(device=DEVICE, img_size=560):
    model = DINOv2Detector(num_classes=len(AFO_CLASSES), img_size=img_size).to(device)
    return model


# Quick parameter count (backbone not loaded yet)
print("Model classes defined.")
print("Call build_sds_model() or build_afo_model() to instantiate.")

Model classes defined.
Call build_sds_model() or build_afo_model() to instantiate.


## 6. Loss — Hungarian Matching (DETR-style)

DETR uses **bipartite matching** to pair predictions with ground truth without NMS. The total loss is a weighted sum of classification cross-entropy and GIoU + L1 box regression.

In [19]:
def box_cxcywh_to_xyxy(boxes):
    """Convert (cx,cy,w,h) → (x1,y1,x2,y2)."""
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], dim=-1)


def generalised_iou(pred_xyxy, tgt_xyxy):
    """GIoU between two sets of boxes. Inputs: [N,4], [M,4] → [N,M]."""
    # Intersection
    inter_x1 = torch.max(pred_xyxy[:, None, 0], tgt_xyxy[None, :, 0])
    inter_y1 = torch.max(pred_xyxy[:, None, 1], tgt_xyxy[None, :, 1])
    inter_x2 = torch.min(pred_xyxy[:, None, 2], tgt_xyxy[None, :, 2])
    inter_y2 = torch.min(pred_xyxy[:, None, 3], tgt_xyxy[None, :, 3])
    inter_w  = (inter_x2 - inter_x1).clamp(0)
    inter_h  = (inter_y2 - inter_y1).clamp(0)
    inter    = inter_w * inter_h

    pred_area = (pred_xyxy[:, 2] - pred_xyxy[:, 0]).clamp(0) * \
                (pred_xyxy[:, 3] - pred_xyxy[:, 1]).clamp(0)
    tgt_area  = (tgt_xyxy[:, 2]  - tgt_xyxy[:, 0]).clamp(0) * \
                (tgt_xyxy[:, 3]  - tgt_xyxy[:, 1]).clamp(0)
    union  = pred_area[:, None] + tgt_area[None, :] - inter
    iou    = inter / union.clamp(1e-6)

    # Enclosing box
    enc_x1 = torch.min(pred_xyxy[:, None, 0], tgt_xyxy[None, :, 0])
    enc_y1 = torch.min(pred_xyxy[:, None, 1], tgt_xyxy[None, :, 1])
    enc_x2 = torch.max(pred_xyxy[:, None, 2], tgt_xyxy[None, :, 2])
    enc_y2 = torch.max(pred_xyxy[:, None, 3], tgt_xyxy[None, :, 3])
    enc_area = (enc_x2 - enc_x1).clamp(0) * (enc_y2 - enc_y1).clamp(0)

    giou = iou - (enc_area - union) / enc_area.clamp(1e-6)
    return giou


class HungarianMatcher(nn.Module):
    """Finds the optimal bipartite assignment between predictions and targets."""

    def __init__(self, cost_cls: float = 1.0,
                 cost_bbox: float = 5.0,
                 cost_giou: float = 2.0):
        super().__init__()
        self.cost_cls  = cost_cls
        self.cost_bbox = cost_bbox
        self.cost_giou = cost_giou

    @torch.no_grad()
    def forward(self, logits, bboxes, targets):
        """
        Args:
            logits  : [B, Q, C+1]
            bboxes  : [B, Q, 4]  (cx,cy,w,h) normalised
            targets : list of dicts with 'boxes' and 'labels'
        Returns:
            List of (pred_idx, tgt_idx) per batch item
        """
        B, Q = logits.shape[:2]
        probs = logits.softmax(-1)   # [B, Q, C+1]

        indices = []
        for b in range(B):
            tgt_boxes  = targets[b]["boxes"].to(logits.device)   # [T, 4]
            tgt_labels = targets[b]["labels"].to(logits.device)  # [T]
            T = len(tgt_labels)
            if T == 0:
                indices.append((torch.tensor([], dtype=torch.long),
                                torch.tensor([], dtype=torch.long)))
                continue

            # Classification cost: negative probability of target class
            c_cls  = -probs[b][:, tgt_labels]  # [Q, T]

            # L1 box cost
            c_bbox = torch.cdist(bboxes[b], tgt_boxes, p=1)  # [Q, T]

            # GIoU cost
            p_xyxy = box_cxcywh_to_xyxy(bboxes[b])           # [Q, 4]
            t_xyxy = box_cxcywh_to_xyxy(tgt_boxes)           # [T, 4]
            c_giou = -generalised_iou(p_xyxy, t_xyxy)        # [Q, T]

            C = (self.cost_cls  * c_cls  +
                 self.cost_bbox * c_bbox +
                 self.cost_giou * c_giou).cpu().numpy()

            row, col = linear_sum_assignment(C)
            indices.append((
                torch.tensor(row, dtype=torch.long),
                torch.tensor(col, dtype=torch.long),
            ))

        return indices


class DETRLoss(nn.Module):
    """Computes DETR-style set prediction loss."""

    def __init__(self, num_classes: int,
                 eos_coef: float  = 0.1,
                 bbox_coef: float = 5.0,
                 giou_coef: float = 2.0):
        super().__init__()
        self.num_classes = num_classes
        self.matcher     = HungarianMatcher()
        self.bbox_coef   = bbox_coef
        self.giou_coef   = giou_coef

        # Down-weight the "no object" class
        empty_weight = torch.ones(num_classes + 1)
        empty_weight[-1] = eos_coef
        self.register_buffer("empty_weight", empty_weight)

    def forward(self, logits, bboxes, targets):
        """
        Args:
            logits  : [B, Q, C+1]
            bboxes  : [B, Q, 4]
            targets : list of dicts
        Returns:
            loss_dict : dict with individual and total losses
        """
        indices = self.matcher(logits, bboxes, targets)
        B, Q, _ = logits.shape
        device  = logits.device

        # ── Classification loss ────────────────────────────────────────────
        tgt_classes = torch.full((B, Q), self.num_classes,
                                 dtype=torch.long, device=device)
        for b, (pred_idx, tgt_idx) in enumerate(indices):
            if len(pred_idx):
                tgt_classes[b, pred_idx] = targets[b]["labels"][tgt_idx].to(device)

        loss_cls = F.cross_entropy(
            logits.reshape(-1, self.num_classes + 1),
            tgt_classes.reshape(-1),
            weight=self.empty_weight.to(device)
        )

        # ── Box regression loss (only on matched queries) ──────────────────
        loss_bbox = torch.tensor(0.0, device=device)
        loss_giou = torch.tensor(0.0, device=device)
        num_matched = 0

        for b, (pred_idx, tgt_idx) in enumerate(indices):
            if len(pred_idx) == 0:
                continue
            pred_b = bboxes[b, pred_idx]
            tgt_b  = targets[b]["boxes"][tgt_idx].to(device)
            loss_bbox += F.l1_loss(pred_b, tgt_b, reduction="sum")
            p_xyxy = box_cxcywh_to_xyxy(pred_b)
            t_xyxy = box_cxcywh_to_xyxy(tgt_b)
            loss_giou += (1 - generalised_iou(p_xyxy, t_xyxy).diag()).sum()
            num_matched += len(pred_idx)

        n = max(num_matched, 1)
        loss_bbox = loss_bbox / n
        loss_giou = loss_giou / n

        total = loss_cls + self.bbox_coef * loss_bbox + self.giou_coef * loss_giou

        return {
            "loss_cls":  loss_cls,
            "loss_bbox": loss_bbox,
            "loss_giou": loss_giou,
            "loss_total": total,
        }


print("Loss functions defined.")

Loss functions defined.


## 7. Training Loop

In [20]:
def train_one_epoch(model, criterion, dataloader, optimizer, device, epoch):
    model.train()
    running = defaultdict(float)
    pbar = tqdm(dataloader, desc=f"Epoch {epoch}")

    for imgs, targets in pbar:
        imgs = imgs.to(device)
        logits, bboxes = model(imgs)
        losses = criterion(logits, bboxes, targets)

        optimizer.zero_grad()
        losses["loss_total"].backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
        optimizer.step()

        for k, v in losses.items():
            running[k] += v.item()
        pbar.set_postfix(loss=f"{losses['loss_total'].item():.4f}")

    n = len(dataloader)
    return {k: v / n for k, v in running.items()}


@torch.no_grad()
def evaluate(model, criterion, dataloader, device):
    model.eval()
    running = defaultdict(float)
    for imgs, targets in tqdm(dataloader, desc="Eval"):
        imgs   = imgs.to(device)
        logits, bboxes = model(imgs)
        losses = criterion(logits, bboxes, targets)
        for k, v in losses.items():
            running[k] += v.item()
    n = len(dataloader)
    return {k: v / n for k, v in running.items()}


def train(model, train_loader, val_loader,
          num_classes, num_epochs=50,
          lr=1e-4, lr_backbone=1e-5,
          weight_decay=1e-4,
          device=DEVICE,
          checkpoint_dir="./checkpoints"):

    Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
    criterion = DETRLoss(num_classes).to(device)

    # Separate LR for frozen-partial backbone vs head
    backbone_params = list(model.backbone.parameters())
    head_params     = list(model.head.parameters())
    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": lr_backbone},
        {"params": head_params,     "lr": lr},
    ], weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=1e-6)

    history = {"train_loss": [], "val_loss": []}
    best_val = float("inf")

    for epoch in range(1, num_epochs + 1):
        train_metrics = train_one_epoch(
            model, criterion, train_loader, optimizer, device, epoch)
        val_metrics   = evaluate(model, criterion, val_loader, device)
        scheduler.step()

        history["train_loss"].append(train_metrics["loss_total"])
        history["val_loss"].append(val_metrics["loss_total"])

        print(f"[{epoch:03d}/{num_epochs}] "
              f"train={train_metrics['loss_total']:.4f}  "
              f"val={val_metrics['loss_total']:.4f}  "
              f"cls={val_metrics['loss_cls']:.4f}  "
              f"giou={val_metrics['loss_giou']:.4f}")

        # Checkpoint
        if val_metrics["loss_total"] < best_val:
            best_val = val_metrics["loss_total"]
            torch.save(model.state_dict(),
                       f"{checkpoint_dir}/best_model.pt")
            print(f"  ↳ Saved best checkpoint (val={best_val:.4f})")

    return history


print("Training utilities defined.")

Training utilities defined.


## 8. Evaluation — mAP Calculation

We implement per-class AP with IoU@0.50 (mAP50) and IoU@0.50:0.95 (mAP50:95), matching the metrics reported in the MSO-DETR paper.

In [21]:
def box_iou(boxes1_xyxy, boxes2_xyxy):
    """IoU between two sets of boxes [N,4] and [M,4] → [N,M]."""
    x1 = torch.max(boxes1_xyxy[:, None, 0], boxes2_xyxy[None, :, 0])
    y1 = torch.max(boxes1_xyxy[:, None, 1], boxes2_xyxy[None, :, 1])
    x2 = torch.min(boxes1_xyxy[:, None, 2], boxes2_xyxy[None, :, 2])
    y2 = torch.min(boxes1_xyxy[:, None, 3], boxes2_xyxy[None, :, 3])
    inter = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    a1 = (boxes1_xyxy[:, 2] - boxes1_xyxy[:, 0]) * \
         (boxes1_xyxy[:, 3] - boxes1_xyxy[:, 1])
    a2 = (boxes2_xyxy[:, 2] - boxes2_xyxy[:, 0]) * \
         (boxes2_xyxy[:, 3] - boxes2_xyxy[:, 1])
    union = a1[:, None] + a2[None, :] - inter
    return inter / union.clamp(1e-6)


def compute_ap(precision, recall):
    """Compute area under precision-recall curve using 101-point interpolation."""
    recall_interp    = np.linspace(0, 1, 101)
    precision_interp = np.interp(recall_interp,
                                 np.concatenate([[0], recall, [1]]),
                                 np.concatenate([[0], precision, [0]]))
    return np.mean(precision_interp)


@torch.no_grad()
def compute_map(model, dataloader, num_classes, device,
                conf_thresh=0.5, iou_thresh_list=None):
    """
    Compute mAP50 and mAP50:95.

    Args:
        model          : trained DINOv2Detector
        dataloader     : val/test DataLoader
        num_classes    : number of foreground classes
        device         : torch device
        conf_thresh    : minimum confidence to consider a prediction
        iou_thresh_list: list of IoU thresholds for mAP50:95
    Returns:
        dict with mAP50, mAP50_95, per_class_ap
    """
    if iou_thresh_list is None:
        iou_thresh_list = np.arange(0.50, 1.00, 0.05).tolist()

    model.eval()

    # Accumulate per-class: list of (score, tp) pairs per IoU threshold
    # all_results[cls][iou_idx] = list of (score, is_tp)
    all_results = [[[] for _ in iou_thresh_list] for _ in range(num_classes)]
    num_gt      = [0] * num_classes

    for imgs, targets in tqdm(dataloader, desc="Computing mAP"):
        imgs   = imgs.to(device)
        logits, bboxes = model(imgs)
        probs  = logits.softmax(-1)  # [B, Q, C+1]
        scores, labels = probs[..., :-1].max(-1)  # [B, Q]

        B = imgs.shape[0]
        for b in range(B):
            # Filter by confidence
            mask   = scores[b] > conf_thresh
            p_scores = scores[b, mask].cpu()
            p_labels = labels[b, mask].cpu()
            p_boxes  = box_cxcywh_to_xyxy(bboxes[b, mask]).cpu()

            tgt_boxes  = box_cxcywh_to_xyxy(targets[b]["boxes"])  # [T, 4]
            tgt_labels = targets[b]["labels"]                      # [T]

            for c in range(num_classes):
                gt_c   = (tgt_labels == c).nonzero(as_tuple=False).squeeze(1)
                pred_c = (p_labels   == c).nonzero(as_tuple=False).squeeze(1)
                num_gt[c] += len(gt_c)

                if len(gt_c) == 0 or len(pred_c) == 0:
                    for ii in range(len(iou_thresh_list)):
                        for _ in range(len(pred_c)):
                            all_results[c][ii].append((p_scores[pred_c[_]].item(), 0))
                    continue

                ious = box_iou(p_boxes[pred_c], tgt_boxes[gt_c])  # [P, G]
                for ii, iou_th in enumerate(iou_thresh_list):
                    gt_matched = torch.zeros(len(gt_c), dtype=torch.bool)
                    # Sort predictions by descending score
                    order = p_scores[pred_c].argsort(descending=True)
                    for pi in order:
                        best_iou, best_g = ious[pi].max(0)
                        tp = (best_iou >= iou_th) and (not gt_matched[best_g])
                        if tp:
                            gt_matched[best_g] = True
                        all_results[c][ii].append(
                            (p_scores[pred_c[pi]].item(), int(tp)))

    # Compute AP per class per IoU threshold
    ap_matrix = np.zeros((num_classes, len(iou_thresh_list)))

    for c in range(num_classes):
        for ii in range(len(iou_thresh_list)):
            results = sorted(all_results[c][ii], key=lambda x: -x[0])
            if not results:
                continue
            tps = np.cumsum([r[1] for r in results])
            fps = np.cumsum([1 - r[1] for r in results])
            prec = tps / (tps + fps)
            rec  = tps / max(num_gt[c], 1)
            ap_matrix[c, ii] = compute_ap(prec, rec)

    mAP50    = ap_matrix[:, 0].mean()
    mAP50_95 = ap_matrix.mean()

    return {
        "mAP50":       mAP50,
        "mAP50_95":    mAP50_95,
        "per_class_ap": ap_matrix[:, 0],  # AP@0.5 per class
    }


print("mAP evaluation utilities defined.")

mAP evaluation utilities defined.


## 9. Run Training — SeaDronesSee

In [22]:
# ── Instantiate datasets ─────────────────────────────────────────────────────

IMG_SIZE   = 560    # divisible by 14 (DINOv2 patch size) and 56 (RF-DETR)
BATCH_SIZE = 8      # fits comfortably on RTX 4090 at 560px
NUM_EPOCHS = 50

# ── AFO (ready now) ───────────────────────────────────────────────────────────
afo_train_ds = AFODataset(AFO_ROOT, split="train", img_size=IMG_SIZE, augment=True)
afo_val_ds   = AFODataset(AFO_ROOT, split="val",   img_size=IMG_SIZE, augment=False)

afo_train_loader = DataLoader(afo_train_ds, batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=4,
                              collate_fn=sds_collate, pin_memory=True)
afo_val_loader   = DataLoader(afo_val_ds,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=4,
                              collate_fn=sds_collate, pin_memory=True)

print(f"AFO  train: {len(afo_train_ds)} samples")
print(f"AFO  val  : {len(afo_val_ds)} samples")

# ── SeaDronesSee (uncomment once 7z extraction finishes) ─────────────────────
# sds_train_ds = SeaDronesSeeDataset(SDS_ROOT, split="train", img_size=IMG_SIZE, augment=True)
# sds_val_ds   = SeaDronesSeeDataset(SDS_ROOT, split="val",   img_size=IMG_SIZE, augment=False)
# sds_train_loader = DataLoader(sds_train_ds, batch_size=BATCH_SIZE,
#                               shuffle=True,  num_workers=4,
#                               collate_fn=sds_collate, pin_memory=True)
# sds_val_loader   = DataLoader(sds_val_ds,   batch_size=BATCH_SIZE,
#                               shuffle=False, num_workers=4,
#                               collate_fn=sds_collate, pin_memory=True)
# print(f"SDS  train: {len(sds_train_ds)} samples")
# print(f"SDS  val  : {len(sds_val_ds)} samples")


Training cells ready — configure dataset paths and uncomment to run.


## 10. Run Training — AFO Dataset

In [23]:
# ── AFO training run ─────────────────────────────────────────────────────────

afo_model = build_afo_model(DEVICE)

total_params     = sum(p.numel() for p in afo_model.parameters())
trainable_params = sum(p.numel() for p in afo_model.parameters() if p.requires_grad)
print(f"Total params    : {total_params/1e6:.1f} M")
print(f"Trainable params: {trainable_params/1e6:.1f} M")

history_afo = train(
    afo_model,
    afo_train_loader,
    afo_val_loader,
    num_classes    = len(AFO_CLASSES),
    num_epochs     = NUM_EPOCHS,
    lr             = 1e-4,
    lr_backbone    = 1e-5,
    device         = DEVICE,
    checkpoint_dir = str(Path.home() / 'checkpoints' / 'afo')
)


AFO training cell ready.


## 11. Training Curve Visualisation

In [24]:
def plot_training_history(history, title="Training History"):
    fig, ax = plt.subplots(figsize=(10, 4))
    epochs = range(1, len(history["train_loss"]) + 1)
    ax.plot(epochs, history["train_loss"], label="Train loss")
    ax.plot(epochs, history["val_loss"],   label="Val loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

# plot_training_history(history, "SeaDronesSee Training")
# plot_training_history(history_afo, "AFO Training")

print("Plotting utility defined.")

Plotting utility defined.


## 12. Evaluation on Test Set

In [25]:
def run_evaluation(model, dataset_root, split, dataset_cls,
                   class_names, batch_size=4, device=DEVICE):

    test_ds = dataset_cls(dataset_root, split=split, augment=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             collate_fn=sds_collate, num_workers=2)

    # Load best checkpoint
    # model.load_state_dict(torch.load("checkpoints/sds/best_model.pt"))

    metrics = compute_map(model, test_loader, len(class_names), device)

    print(f"\n{'─'*45}")
    print(f"  mAP@0.50      : {metrics['mAP50']:.4f}")
    print(f"  mAP@0.50:0.95 : {metrics['mAP50_95']:.4f}")
    print(f"{'─'*45}")
    for cls, ap in zip(class_names, metrics["per_class_ap"]):
        print(f"  {cls:<25} AP@0.50: {ap:.4f}")

    return metrics


# Example usage:
# sds_metrics = run_evaluation(
#     sds_model, SDS_ROOT, "test", SeaDronesSeeDataset, SDS_CLASSES)

# afo_metrics = run_evaluation(
#     afo_model, AFO_ROOT, "test", AFODataset, AFO_CLASSES)

print("Evaluation cell ready.")

Evaluation cell ready.


## 13. Inference & Visualisation

In [26]:
@torch.no_grad()
def predict_single(model, image_path, class_names,
                   img_size=640, conf_thresh=0.5, device=DEVICE):
    """
    Run inference on a single image and display results.

    Args:
        model       : trained DINOv2Detector
        image_path  : path to input image
        class_names : list of class name strings
        img_size    : resize resolution
        conf_thresh : confidence threshold for displaying detections
    """
    normalize = T.Normalize(mean=[0.485, 0.456, 0.406],
                             std =[0.229, 0.224, 0.225])
    model.eval()

    img_orig = Image.open(image_path).convert("RGB")
    img_resized = img_orig.resize((img_size, img_size), Image.BILINEAR)
    img_t = normalize(T.functional.to_tensor(img_resized)).unsqueeze(0).to(device)

    logits, bboxes = model(img_t)       # [1, Q, C+1], [1, Q, 4]
    probs  = logits.softmax(-1)[0]      # [Q, C+1]
    scores, pred_labels = probs[:, :-1].max(-1)

    mask = scores > conf_thresh
    det_scores = scores[mask].cpu()
    det_labels = pred_labels[mask].cpu()
    det_boxes  = bboxes[0, mask].cpu()  # (cx,cy,w,h)

    # Visualise
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    img_np = np.array(img_resized) / 255.0
    H, W = img_np.shape[:2]
    COLORS = plt.cm.get_cmap("tab10", len(class_names))

    for ax_idx, (ax_img, title) in enumerate(
            zip([img_np.copy(), img_np.copy()], ["Original", "Detections"])):
        axes[ax_idx].imshow(ax_img)
        axes[ax_idx].set_title(title, fontsize=12)
        axes[ax_idx].axis("off")
        if ax_idx == 1:
            for box, label, score in zip(det_boxes, det_labels, det_scores):
                cx, cy, w, h = box.numpy()
                x1 = (cx - w/2) * W
                y1 = (cy - h/2) * H
                color = COLORS(label.item())
                rect  = patches.Rectangle(
                    (x1, y1), w*W, h*H,
                    linewidth=2, edgecolor=color, facecolor="none")
                axes[ax_idx].add_patch(rect)
                axes[ax_idx].text(
                    x1, y1 - 5,
                    f"{class_names[label.item()]} {score:.2f}",
                    color=color, fontsize=8, weight="bold",
                    bbox=dict(boxstyle="round,pad=0.1", fc="black", alpha=0.4))

    plt.suptitle(f"{Path(image_path).name}  —  {len(det_scores)} detections",
                 fontsize=13)
    plt.tight_layout()
    plt.show()

    return {"boxes": det_boxes, "labels": det_labels, "scores": det_scores}


# Example:
# results = predict_single(sds_model, "data/seadronessee/images/test/001.jpg",
#                          SDS_CLASSES, conf_thresh=0.4)

print("Inference utility defined.")

Inference utility defined.


## 14. Comparison Summary Table

Reference numbers from the MSO-DETR paper (Table 4) plus placeholder rows for all three paths in this notebook.
Fill in your results as you complete each section: RF-DETR (§0A), DINOv3 (§0B), DINOv2 (§3–13).

In [27]:
import pandas as pd

paper_results = {
    "Model": [
        "Faster R-CNN", "YOLOv5m", "YOLOv7",
        "YOLOv8s", "YOLOv8m", "YOLOv11m",
        "RT-DETR-R18 (baseline)", "MSO-DETR (paper)",
        # ── Section 0A ──────────────────────────────────────────
        "RF-DETR-Base (§0A)",      # ← fill in after fine-tuning
        "RF-DETR-Large (§0A)",     # ← optional large variant
        # ── Section 0B ──────────────────────────────────────────
        "DINOv3-ViT-S + DETR (§0B)",  # ← fill in after training
        "DINOv3-ViT-B + DETR (§0B)",  # ← optional larger backbone
        # ── Sections 1–15 ───────────────────────────────────────
        "DINOv2-ViT-S + DETR (§3)",   # ← fill in after training
    ],
    "mAP50 (%)": [
        56.2, 71.4, 72.3, 70.8, 73.7, 73.6, 82.1, 82.0,
        "—", "—",   # RF-DETR
        "—", "—",   # DINOv3
        "—",        # DINOv2
    ],
    "mAP50:95 (%)": [
        35.3, 41.8, 43.6, 42.5, 44.5, 44.3, 48.4, 48.9,
        "—", "—",
        "—", "—",
        "—",
    ],
    "Params (M)": [
        63.2, 21.2, 37.1, 10.6, 25.8, 20.0, 19.9, 6.5,
        29.0, 128.0,    # RF-DETR Base / Large
        "~22+head", "~86+head",  # DINOv3 ViT-S / ViT-B
        "~22+head",              # DINOv2 ViT-S
    ],
    "Notes": [
        "", "", "", "", "", "", "", "",
        "pip install rfdetr", "pip install rfdetr",
        "timm>=1.0.20; Gram anchoring", "timm>=1.0.20; Gram anchoring",
        "torch.hub dinov2_vits14",
    ],
}

df = pd.DataFrame(paper_results)
print("SeaDronesSee Benchmark — from MSO-DETR paper (Table 4)")
print("=" * 80)
print(df.to_string(index=False))

SeaDronesSee Benchmark — from MSO-DETR paper (Table 4)
                    Model mAP50 (%) mAP50:95 (%) Params (M)                        Notes
             Faster R-CNN      56.2         35.3       63.2                             
                  YOLOv5m      71.4         41.8       21.2                             
                   YOLOv7      72.3         43.6       37.1                             
                  YOLOv8s      70.8         42.5       10.6                             
                  YOLOv8m      73.7         44.5       25.8                             
                 YOLOv11m      73.6         44.3       20.0                             
   RT-DETR-R18 (baseline)      82.1         48.4       19.9                             
         MSO-DETR (paper)      82.0         48.9        6.5                             
       RF-DETR-Base (§0A)         —            —       29.0           pip install rfdetr
      RF-DETR-Large (§0A)         —            —      1

## 15. Next Steps

| Idea | Relevant section | Description |
|------|-----------------|-------------|
| **Swap RF-DETR backbone to DINOv3** | §0A | RF-DETR ships with DINOv2; once Roboflow merges timm DINOv3 support, passing `backbone='dinov3'` will give an instant bump. |
| **DHSA integration** | §0B / §3 | Port Dynamic-range Histogram Self-Attention from MSO-DETR into the encoder (replaces standard self-attention in AIFI). |
| **get_intermediate_layers API** | §0B | Replace manual block tapping with timm's built-in `get_intermediate_layers` for cleaner multi-scale extraction and easier model updates. |
| **Maritime domain pre-fine-tuning** | §0B / §3 | Fine-tune DINOv3 on open-ocean UAV images before plugging into the detector to close the domain gap vs ImageNet pre-training. |
| **ONNX / TensorRT export** | §0A | RF-DETR has built-in `model.export()`. For the custom builds use `torch.onnx.export` then `trtexec` on RHEL. |
| **Sea-specific augmentations** | §0A / §3 | Add Mosaic, MixUp, glare simulation, wave texture overlay to harden against maritime conditions. |
| **INT8 PTQ** | §0B / §3 | Apply per-channel symmetric quantisation to backbone projection layers for edge deployment on constrained RHEL hardware. |

In [28]:
# Quick DINOv2 smoke test (no dataset needed)
print("Running smoke test (loads DINOv2 weights — needs internet first time)...")

# dummy_backbone = DINOv2Backbone("vits14", out_channels=256, img_size=224)
# dummy_backbone.eval()
# x = torch.randn(1, 3, 224, 224)
# with torch.no_grad():
#     p3, p4, p5 = dummy_backbone(x)
# print(f"P3: {tuple(p3.shape)}  P4: {tuple(p4.shape)}  P5: {tuple(p5.shape)}")

print("Uncomment the smoke test to verify backbone output shapes.")

Running smoke test (loads DINOv2 weights — needs internet first time)...
Uncomment the smoke test to verify backbone output shapes.
